In [10]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import Literal

_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

client = OpenAI()

In [11]:
SYSTEM_PROMPT = """You are a friendly, practical AI travel advisor.
Help the user plan a personalized, realistic trip through conversation.
Before generating an itinerary, make sure you know all three essentials:
(1) destination, 
(2) trip duration (number of days or dates), and 
(3) at least one travel interest (for example food, museums, hiking, beaches, 
nightlife, or shopping). 
 
Ask one concise, natural follow-up covering only the
missing essentials. Do not ask for optional preferences if the user has not
volunteered them; use sensible assumptions instead.

If the destination, dates, duration, or other details conflict or are materially
ambiguous, ask the user to clarify before planning. Do not silently choose between
contradictory details. Treat an explicit "surprise me" destination as permission
to suggest a destination and proceed, while briefly stating your choice.

Once the essentials are clear and there is no unresolved contradiction, 
provide the complete plan without asking for confirmation. 
Include: 
a memorable trip title; 
a short summary reflecting the user's interests and stated preferences;
a day-by-day itinerary with a sensible pace and geographically coherent activities; 
practical travel tips; 
and an estimated budget with a stated currency,

approximate per-person total, and a clear breakdown 
(such as lodging, food, local transport, and activities). 

State assumptions, especially when dates, departure city,
or budget are unknown. 

Keep estimates approximate, avoid claiming
live availability or current opening hours, and suggest checking details
before booking.

Respect any accessibility, dietary, safety, or budget constraints the user mentions.
Be transparent when a specific recommendation or cost is uncertain.
"""

Token totals are accumulated from each API response's usage data. The cost is an estimate using the Standard `gpt-6-luna` rates listed in the [OpenAI API pricing guide](https://developers.openai.com/api/docs/pricing): $0.10 per 1M input tokens, $0.01 per 1M cached input tokens, and $0.50 per 1M output tokens. Update the rates if you change the model or pricing tier.

In [12]:
MODEL = "gpt-6-luna"

PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50}
}
history = [{"role": "system", "content": SYSTEM_PROMPT}]

conversation_usage = {
    "input_tokens": 0,
    "cached_input_tokens": 0,
    "output_tokens": 0,
    "estimated_cost_usd": 0.0,
}

class Structure(BaseModel):

    title: str = Field(description="A title for the trip")
                       
    summary: str = Field(description= "Trip Summary")

    itinerary: str = Field(description= "Day by Day itinerary")  

    practical_tips: str = Field(descroption= "Practical travel tips")

    budget: str = Field(description = "Estimated buget")
  

def format_as_markdown(trip: Structure) -> str:
    return f"""# {trip.title}

## Summary
{trip.summary}

## Day-by-day itinerary
{trip.itinerary}

## Practical tips
{trip.practical_tips}

## Estimated budget
{trip.budget}"""


def send_request(input, **kwargs):
    """Send one Responses API request using the unit's shared model."""
    return client.responses.parse(model=MODEL, input=input, text_format = Structure ,**kwargs)


def chat(user_text):
    history.append({"role": "user", "content": user_text})
    response = send_request(history)
    answer = response.output_text
    history.append({"role": "assistant", "content": answer})

    usage = response.usage
    input_tokens = usage.input_tokens
    cached_input_tokens = getattr(
        getattr(usage, "input_tokens_details", None), "cached_tokens", 0
    ) or 0
    output_tokens = usage.output_tokens

    conversation_usage["input_tokens"] += input_tokens
    conversation_usage["cached_input_tokens"] += cached_input_tokens
    conversation_usage["output_tokens"] += output_tokens

    rates = PRICING_USD_PER_MILLION[MODEL]
    uncached_input_tokens = input_tokens - cached_input_tokens
    conversation_usage["estimated_cost_usd"] += (
        uncached_input_tokens * rates["input"]
        + cached_input_tokens * rates["cached_input"]
        + output_tokens * rates["output"]
    ) / 1_000_000

    print(
        f"Conversation totals: {conversation_usage['input_tokens']:,} input tokens, "
        f"{conversation_usage['output_tokens']:,} output tokens; "
        f"estimated cost ${conversation_usage['estimated_cost_usd']:.6f} USD"
    )
    return format_as_markdown(response.output_parsed)

/var/folders/8z/s15bhnzx7zng08dcc75mh3_c0000gn/T/ipykernel_56960/4132071508.py:23: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descroption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  practical_tips: str = Field(descroption= "Practical travel tips")


In [13]:
from IPython.display import Markdown, display

display(Markdown(chat("I want to visit Barcelona for 3 days. I love food and architecture.")))
display(Markdown(chat("What should I do on the second day?")))

Conversation totals: 485 input tokens, 832 output tokens; estimated cost $0.000465 USD


# Barcelona: Modernist Masterpieces & Catalan Flavours

## Summary
A three-day city break pairing Barcelona’s Gothic and Modernist architecture with tapas, market snacks, and Catalan cooking. Assumptions: dates, departure city, and budget are unspecified; the plan is for three full days, with flights excluded.

## Day-by-day itinerary
Day 1 — Gothic Quarter & El Born: Explore the narrow lanes and historic architecture of the Gothic Quarter, then continue to El Born and the exterior of Santa Maria del Mar. Take a relaxed lunch of tapas or a menú del día, and browse the Santa Caterina Market for local produce and a snack. In the evening, enjoy a Catalan meal in El Born; reserve ahead for a popular restaurant.

Day 2 — Gaudí & the Eixample: Visit the Sagrada Família (book timed entry in advance), then walk through the Eixample to admire Casa Milà and Casa Batlló. Choose one interior visit to keep the day comfortable and your ticket costs manageable. Pause for lunch nearby, perhaps trying pa amb tomàquet and seasonal Catalan dishes. Finish with a relaxed dinner in the neighborhood.

Day 3 — Park Güell & Gràcia: Start at Park Güell, another major Gaudí site; check ticket requirements and reserve ahead. Head downhill to Gràcia for a leisurely lunch and a wander through its lively squares and smaller streets. Spend the afternoon sampling pastries or vermouth, or visit another architectural landmark that interests you. Keep the evening open for a final tapas meal.

## Practical tips
Barcelona is very walkable, but comfortable shoes help on its hills and cobbled streets; use the metro for longer hops. Keep bags secure in crowded areas and on public transport. Book the Sagrada Família, Park Güell, and any chosen house interiors in advance, and check official sites for current prices, entry rules, and opening times. Markets can be busy and some are more geared toward visitors; look for seasonal produce and compare prices. Restaurant menus and service vary, so check the menu before sitting down.

## Estimated budget
Approximate per-person estimate for 3 days / 2 nights: €420–€715, excluding flights and assuming a shared double room. Lodging: €180–€320; food: €150–€240; local transport: €20–€35; paid sights and activities: €70–€120. Costs vary with travel dates, accommodation choice, and how many architectural interiors you visit; add a buffer for drinks, snacks, or shopping.

Conversation totals: 1,529 input tokens, 1,319 output tokens; estimated cost $0.000812 USD


# Day 2 in Barcelona: Gaudí & the Eixample

## Summary
A full but manageable day focused on Barcelona’s signature Modernist architecture, with time for a Catalan lunch and a relaxed evening.

## Day-by-day itinerary
Morning: Start at the Sagrada Família. Allow around 1.5–2 hours to explore, and book timed admission in advance. Afterwards, walk or take the metro toward Passeig de Gràcia.

Lunch: Stop in the Eixample for a menú del día or a relaxed Catalan lunch. Try pa amb tomàquet, seasonal vegetables, or a rice dish.

Afternoon: See Casa Milà (La Pedrera) and Casa Batlló from the outside along Passeig de Gràcia. To avoid overloading the day, choose one for an interior visit; Casa Milà is a good option if you’re especially interested in architecture and rooftop details. Allow about 1.5–2 hours for a visit, depending on the ticket and pace.

Evening: Take a break at your hotel or enjoy a drink nearby, then have dinner in the Eixample. Pick a restaurant with a menu that appeals to you and reserve ahead if you have a particular place in mind.

## Practical tips
The Sagrada Família and major house museums often use timed tickets, so check official websites and book ahead. Wear comfortable shoes and leave some flexibility between visits. If you prefer a lighter day, skip the interior visit to Casa Milà or Casa Batlló and enjoy the architecture from the street instead. Check current entry details and opening times before you go.

## Estimated budget
Approximate per-person day cost, excluding lodging and flights: €75–€160. Food: €35–€65; local transport: €5–€10; admissions: approximately €35–€85 depending on which interiors you visit. Prices vary by ticket type and date, so verify costs before booking.